# 02 | Which sources and keys can Finance trust? (Q1)

*BDCC project, Group G. Run `01_land_raw` first.*

1. What does one row mean in each source (grain, key, key scope)?
2. What happens to row counts and distinct keys across one consequential join?
3. Which source do we exclude, and on what evidence?

The numbers here describe the whole release, without a cutoff. They justify the design. The close itself only uses what is available by each cutoff (notebook 03).

**Where this fits in the brief**

- Answers **Q1**: grain and scope of the keys, row counts and distinct keys before and after one consequential join, and one source exclusion with its evidence.
- **Configuration A** asks us to profile identifiers and establish the meaning of a row on the development extract. Running the same notebook on the full release (**configuration B**, widget `schema = bdcc_g_full`) shows the same patterns at scale.
- Every result we cite in the report is saved as a `q1_*` table. Working steps are temporary views (P1: temporary views for steps, named tables for results).

In [ ]:
import re

# Same parameters as notebook 01, so both notebooks always look at the same release.
dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("schema", "bdcc_g_dev")
dbutils.widgets.text("validator_run", "dev_initial")
catalog = dbutils.widgets.get("catalog").strip()
schema = dbutils.widgets.get("schema").strip()
validator_run = dbutils.widgets.get("validator_run").strip()
for value in (catalog, schema):
    assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", value), "Use a simple catalog and schema identifier"
assert re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_-]*", validator_run), "Use letters, digits, dash or underscore"
spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"USE SCHEMA {schema}")
spark.sql("SET TIME ZONE 'UTC'")
validator_path = f"/Volumes/{catalog}/{schema}/landing/validator/{validator_run}"

# All eight candidate sources are profiled, including the two we exclude later: an exclusion
# needs evidence too.
SOURCES = ["gateway_requests", "processor_events", "refund_events", "settlement_lines", "reference_mappings",
           "terminal_assignments", "merchant_campaigns", "terminal_service_tickets"]

## 1. What does one row mean?

A source_row_id identifies a transport row, not a business fact. The data dictionary says so. A business fact is identified by its key, which always includes the system that issued it, plus a version where the source is versioned.

For each source, the profile below counts the following among the usable rows:

- **replays.** Rows that repeat a fact already delivered: same key, same version, same business content.
- **distinct_facts.** The rows that remain once the replays are removed.
- **distinct_keys and distinct_bare_keys.** The number of keys with and without their namespace. If the second number is smaller, the same identifier is issued by more than one system.
- **multi_version_keys.** Keys that were delivered in more than one version.
- **conflict_groups.** Cases where one key and version carries two different contents.

In [ ]:
# One profile row per source, built from facts_<source> (defined below: usable rows with
# identical repeats collapsed).
#   replays            = usable rows minus distinct facts
#   distinct_keys      = keys WITH their issuing system
#   distinct_bare_keys = the same keys WITHOUT the system. A smaller number means several
#                        systems reuse the same identifier, so the system must be part of the key
#   multi_version_keys = facts that were corrected at least once
#   conflict_groups    = one key and version with more than one content
def profile_sql(name):
    return f'''
SELECT '{name}' AS source_table,
       (SELECT count(*) FROM stg_{name}) AS delivered,
       (SELECT count(*) FROM stg_{name} WHERE validation_issue IS NOT NULL) AS rejected,
       (SELECT count(*) FROM stg_{name} WHERE validation_issue IS NULL) - (SELECT count(*) FROM facts_{name}) AS replays,
       (SELECT count(*) FROM facts_{name}) AS distinct_facts,
       (SELECT count(DISTINCT business_key) FROM facts_{name}) AS distinct_keys,
       (SELECT count(DISTINCT substring_index(business_key, '|', -1)) FROM facts_{name}) AS distinct_bare_keys,
       (SELECT count(*) FROM (SELECT business_key FROM facts_{name}
                              GROUP BY business_key HAVING count(DISTINCT version_no) > 1) k) AS multi_version_keys,
       (SELECT coalesce(max(version_no), 0) FROM facts_{name}) AS max_version,
       (SELECT count(*) FROM (SELECT business_key, version_no FROM facts_{name}
                              GROUP BY business_key, version_no HAVING count(*) > 1) c) AS conflict_groups,
       (SELECT coalesce(sum(n), 0) FROM (SELECT count(*) AS n FROM facts_{name}
                                        GROUP BY business_key, version_no HAVING count(*) > 1) c) AS conflict_rows
'''

# facts_<source>: GROUP BY key, version and content hash collapses identical repeats into one row.
# WHY a temporary view: it is a working step of this notebook, not a result.
for name in SOURCES:
    spark.sql(f'''
CREATE OR REPLACE TEMP VIEW facts_{name} AS
SELECT business_key, version_no, payload_hash
FROM stg_{name}
WHERE validation_issue IS NULL
GROUP BY business_key, version_no, payload_hash
''')
# The profile is a result we cite in the report, so it is saved as a table.
spark.sql("CREATE OR REPLACE TABLE q1_source_profile AS\n" + "\nUNION ALL\n".join(profile_sql(n) for n in SOURCES))

In [ ]:
%sql
-- Evidence for "grain and scope of the keys": compare distinct_keys with distinct_bare_keys.
SELECT * FROM q1_source_profile ORDER BY source_table

In [ ]:
# The Docker validator computed the same profile with different code. Both must agree.
expected = {row["source_table"]: row for row in
            spark.read.option("header", True).option("inferSchema", False)
            .csv(f"{validator_path}/source_profile.csv").collect()}
# WHY column by column and source by source: an agreeing grand total could hide two errors that
# cancel out.
columns = ["delivered", "rejected", "replays", "distinct_facts", "distinct_keys", "distinct_bare_keys",
           "multi_version_keys", "max_version", "conflict_groups", "conflict_rows"]
# A failing assert stops the notebook and names the source and the column that differ.
for row in spark.sql("SELECT * FROM q1_source_profile").collect():
    for column in columns:
        assert int(row[column]) == int(expected[row["source_table"]][column]), (row["source_table"], column)
print("The source profile reconciles with the validator for", len(expected), "sources.")

**Why the scope matters.** The same `gateway_reference` exists in more than one `gateway_system`, for different payments. A key without its namespace would merge two customers' payments into one.

In [ ]:
%sql
-- The same gateway_reference in more than one gateway_system, with two example payments side by
-- side. different_payments > 1 proves these are different payments, not replays.
SELECT gateway_reference,
       count(DISTINCT gateway_system) AS gateway_systems,
       count(DISTINCT payload_hash) AS different_payments,
       min(concat_ws(' ', gateway_system, request_timestamp, currency_cd, requested_amount_minor)) AS one_payment,
       max(concat_ws(' ', gateway_system, request_timestamp, currency_cd, requested_amount_minor)) AS another_payment
FROM stg_gateway_requests
WHERE validation_issue IS NULL
GROUP BY gateway_reference
HAVING count(DISTINCT gateway_system) > 1
ORDER BY gateway_reference
LIMIT 5

**A replay and a conflict look alike until you compare the content.** Both repeat a key and version. The replay repeats the amount too and counts once. The conflict states two amounts for one fact, and nobody can say which is right.

In [ ]:
%sql
-- Rows that repeat a key and version. One content = replay (counts once).
-- More than one content = same-version conflict (goes to quarantine in notebook 03).
SELECT s.business_key, s.version_no, s.source_row_id, s.delivery_id, s.event_type, s.amount_minor,
       CASE WHEN g.payloads = 1 THEN 'replay' ELSE 'same-version conflict' END AS finding
FROM stg_processor_events s
JOIN (SELECT business_key, version_no, count(DISTINCT payload_hash) AS payloads, count(*) AS n_rows
      FROM stg_processor_events WHERE validation_issue IS NULL
      GROUP BY business_key, version_no HAVING count(*) > 1) g
  ON g.business_key = s.business_key AND g.version_no = s.version_no
WHERE s.validation_issue IS NULL
ORDER BY finding, s.business_key, s.source_row_id
LIMIT 8

## 2. One consequential join: payment to acquiring bank

Finance signs off by bank. A payment gets its bank from the terminal assignment **effective at request time**. A terminal has more than one assignment over time, so the join key alone is not enough.

Predict before running: what happens to the row count and to the summed amount when the time condition is left out?

In [ ]:
# q1_payments: one row per payment. DISTINCT removes replays (identical content), so a replayed
# request is not counted twice.
spark.sql('''
CREATE OR REPLACE TEMP VIEW q1_payments AS
SELECT DISTINCT business_key, terminal_system, terminal_id,
       p_request_timestamp AS request_ts, p_requested_amount_minor AS amount_minor
FROM stg_gateway_requests
WHERE validation_issue IS NULL
''')
# q1_assignments: one row per assignment interval of a terminal.
spark.sql('''
CREATE OR REPLACE TEMP VIEW q1_assignments AS
SELECT DISTINCT terminal_system, terminal_id, assignment_id, merchant_id, acquiring_bank_id, p_valid_from, p_valid_to
FROM stg_terminal_assignments
WHERE validation_issue IS NULL
''')
# Five steps, each with rows, distinct payment keys and the summed amount:
#   1 as delivered   2 replays removed   3 WRONG join (terminal only)
#   4 RIGHT join (terminal + interval valid at request time)   5 payments without a bank (must be 0)
# WHY keep the wrong join in the table: it shows that the distinct keys stay the same while rows
# and money double. A check on keys alone would not notice the error.
# Interval rule from the dictionary: valid_from is inclusive (>=), valid_to is exclusive (<).
# The amount mixes EUR and GBP on purpose: here it is a control figure for cardinality, not a
# financial total.
spark.sql('''
CREATE OR REPLACE TABLE q1_join_cardinality AS
SELECT 1 AS step, 'gateway_requests as delivered' AS population,
       count(*) AS n_rows, count(DISTINCT business_key) AS distinct_payment_keys,
       sum(p_requested_amount_minor) AS amount_minor
FROM stg_gateway_requests
UNION ALL
SELECT 2, 'payments: usable rows, a replay counts once',
       count(*), count(DISTINCT business_key), sum(amount_minor)
FROM q1_payments
UNION ALL
SELECT 3, 'payments JOIN assignments ON terminal only',
       count(*), count(DISTINCT p.business_key), sum(p.amount_minor)
FROM q1_payments p
JOIN q1_assignments a ON a.terminal_system = p.terminal_system AND a.terminal_id = p.terminal_id
UNION ALL
SELECT 4, 'payments JOIN the assignment effective at request time',
       count(*), count(DISTINCT p.business_key), sum(p.amount_minor)
FROM q1_payments p
JOIN q1_assignments a ON a.terminal_system = p.terminal_system AND a.terminal_id = p.terminal_id
 AND p.request_ts >= a.p_valid_from AND p.request_ts < a.p_valid_to
UNION ALL
SELECT 5, 'payments with no assignment effective at request time',
       count(*), count(DISTINCT p.business_key), sum(p.amount_minor)
FROM q1_payments p
WHERE NOT EXISTS (SELECT 1 FROM q1_assignments a
                  WHERE a.terminal_system = p.terminal_system AND a.terminal_id = p.terminal_id
                    AND p.request_ts >= a.p_valid_from AND p.request_ts < a.p_valid_to)
''')

In [ ]:
%sql
-- Read as before and after: step 3 doubles n_rows and amount_minor, step 4 is back at step 2.
SELECT * FROM q1_join_cardinality ORDER BY step

Read the table as a before and after:

- step 1 to 2: transport repeats leave, the distinct keys stay
- step 2 to 3: the key-only join multiplies rows and money while the distinct keys do not move. A total built on it would be wrong and would still look plausible.
- step 3 to 4: the effective-date condition (`valid_from` inclusive, `valid_to` exclusive) returns exactly one row per payment
- step 5 must be zero, otherwise a payment has no bank

The assignments must also never overlap, or step 4 would return two banks for one payment:

In [ ]:
%sql
-- Self-join: two different assignments of one terminal whose intervals overlap. Must be 0,
-- otherwise the time condition would return two banks for one payment.
-- b.assignment_id > a.assignment_id counts each pair once.
SELECT count(*) AS overlapping_assignment_pairs
FROM q1_assignments a
JOIN q1_assignments b
  ON b.terminal_system = a.terminal_system AND b.terminal_id = a.terminal_id
 AND b.assignment_id > a.assignment_id
 AND a.p_valid_from < b.p_valid_to AND b.p_valid_from < a.p_valid_to

**The second join that needs its scope: provider posting to payment.** A provider reference is only meaningful inside its provider system. `reference_mappings` translates (`from_system`, `from_reference`) into a gateway payment. Joining on the reference digits alone matches other systems' payments too.

In [ ]:
# q1_postings: one row per provider posting (replays removed).
spark.sql('''
CREATE OR REPLACE TEMP VIEW q1_postings AS
SELECT DISTINCT business_key, provider_system, provider_reference, p_signed_amount_minor AS signed_amount_minor
FROM stg_settlement_lines
WHERE validation_issue IS NULL
''')
# q1_mappings: one row per mapping, (from_system, from_reference) -> gateway payment.
spark.sql('''
CREATE OR REPLACE TEMP VIEW q1_mappings AS
SELECT DISTINCT from_system, from_reference, gateway_system, gateway_reference
FROM stg_reference_mappings
WHERE validation_issue IS NULL
''')
# The same before-and-after idea for the second money join, posting -> payment:
#   2 join on the reference only (WRONG: also matches other systems' references)
#   3 join on issuing system + reference (RIGHT)
#   4 postings that no mapping links anywhere in the release = provider-only candidates,
#     which the close keeps visible as UNLINKED
spark.sql('''
CREATE OR REPLACE TABLE q1_link_cardinality AS
SELECT 1 AS step, 'provider postings: usable rows, a replay counts once' AS population,
       count(*) AS n_rows, count(DISTINCT business_key) AS distinct_posting_keys,
       sum(signed_amount_minor) AS signed_amount_minor
FROM q1_postings
UNION ALL
SELECT 2, 'postings JOIN mappings ON reference only',
       count(*), count(DISTINCT p.business_key), sum(p.signed_amount_minor)
FROM q1_postings p
JOIN q1_mappings m ON m.from_reference = p.provider_reference
UNION ALL
SELECT 3, 'postings JOIN mappings ON issuing system and reference',
       count(*), count(DISTINCT p.business_key), sum(p.signed_amount_minor)
FROM q1_postings p
JOIN q1_mappings m ON m.from_system = p.provider_system AND m.from_reference = p.provider_reference
UNION ALL
SELECT 4, 'postings with no mapping anywhere in the release',
       count(*), count(DISTINCT p.business_key), sum(p.signed_amount_minor)
FROM q1_postings p
WHERE NOT EXISTS (SELECT 1 FROM q1_mappings m
                  WHERE m.from_system = p.provider_system AND m.from_reference = p.provider_reference)
''')

In [ ]:
%sql
-- Step 2 against step 3 shows the fan-out of a join without the issuing system.
SELECT * FROM q1_link_cardinality ORDER BY step

Step 4 is the provider-only population: postings that no mapping links to a payment, whenever we look. More postings are unlinked at C1 than here, because some mappings arrive after C1. That is a cutoff effect and belongs to the close.

Do the sources we keep cover each other?

In [ ]:
%sql
-- Coverage between the sources we keep: how many references on one side find their partner on
-- the other. n_total = n_covered means full coverage. A gap is a population the close must
-- handle explicitly (for example postings without a mapping become UNLINKED).
SELECT 'processor references with a mapping' AS coverage, count(*) AS n_total,
       sum(CASE WHEN m.from_reference IS NOT NULL THEN 1 ELSE 0 END) AS n_covered
FROM (SELECT DISTINCT processor_system, processor_reference FROM stg_processor_events WHERE validation_issue IS NULL) p
LEFT JOIN q1_mappings m ON m.from_system = p.processor_system AND m.from_reference = p.processor_reference
UNION ALL
SELECT 'provider references with a mapping', count(*),
       sum(CASE WHEN m.from_reference IS NOT NULL THEN 1 ELSE 0 END)
FROM (SELECT DISTINCT provider_system, provider_reference FROM q1_postings) p
LEFT JOIN q1_mappings m ON m.from_system = p.provider_system AND m.from_reference = p.provider_reference
UNION ALL
SELECT 'mappings that point to a delivered gateway payment', count(*),
       sum(CASE WHEN g.business_key IS NOT NULL THEN 1 ELSE 0 END)
FROM q1_mappings m
LEFT JOIN (SELECT DISTINCT business_key FROM q1_payments) g
  ON g.business_key = concat_ws('|', m.gateway_system, m.gateway_reference)
UNION ALL
SELECT 'refunds that point to a delivered capture', count(*),
       sum(CASE WHEN c.business_key IS NOT NULL THEN 1 ELSE 0 END)
FROM (SELECT DISTINCT processor_system, capture_event_id FROM stg_refund_events WHERE validation_issue IS NULL) r
LEFT JOIN (SELECT DISTINCT business_key FROM stg_processor_events
           WHERE validation_issue IS NULL AND event_type = 'CAPTURE') c
  ON c.business_key = concat_ws('|', r.processor_system, r.capture_event_id)
UNION ALL
SELECT 'reversals that point to a delivered capture', count(*),
       sum(CASE WHEN c.business_key IS NOT NULL THEN 1 ELSE 0 END)
FROM (SELECT DISTINCT processor_system, related_event_id FROM stg_processor_events
      WHERE validation_issue IS NULL AND event_type = 'REVERSAL') r
LEFT JOIN (SELECT DISTINCT business_key FROM stg_processor_events
           WHERE validation_issue IS NULL AND event_type = 'CAPTURE') c
  ON c.business_key = concat_ws('|', r.processor_system, r.related_event_id)

## 3. One source exclusion and its evidence

`merchant_campaigns` and `terminal_service_tickets` describe merchants and terminals, not money movements. Neither has a payment key or an amount in minor units. The only path from a payment to them runs through the terminal or its merchant, where one payment meets several campaigns or tickets.

The test: attach each of them to the correctly attributed payments and watch the rows and the total.

In [ ]:
# The exclusion test: start from correctly attributed payments (step 1), then attach each
# candidate the only way possible (step 2 through the merchant, step 3 through the terminal).
# If rows and amount grow while the distinct payment keys stay, the source multiplies payments
# and cannot be part of a per-payment close.
# regexp_replace ACQ -> MER: the dictionary places merchant ids in "the matching MERxx namespace
# for the bundle", so terminal namespace ACQnn pairs with merchant namespace MERnn. This pairing
# is an assumption taken from the dictionary.
spark.sql('''
CREATE OR REPLACE TABLE q1_source_exclusion AS
WITH attributed AS (
  SELECT p.business_key, p.amount_minor, p.terminal_system, p.terminal_id, a.merchant_id
  FROM q1_payments p
  JOIN q1_assignments a ON a.terminal_system = p.terminal_system AND a.terminal_id = p.terminal_id
   AND p.request_ts >= a.p_valid_from AND p.request_ts < a.p_valid_to
)
SELECT 1 AS step, 'payments with their effective assignment' AS variant,
       count(*) AS n_rows, count(DISTINCT business_key) AS distinct_payment_keys, sum(amount_minor) AS amount_minor
FROM attributed
UNION ALL
SELECT 2, 'plus merchant_campaigns ON merchant',
       count(*), count(DISTINCT b.business_key), sum(b.amount_minor)
FROM attributed b
JOIN stg_merchant_campaigns c
  ON c.merchant_system = regexp_replace(b.terminal_system, '^ACQ', 'MER') AND c.merchant_id = b.merchant_id
UNION ALL
SELECT 3, 'plus terminal_service_tickets ON terminal',
       count(*), count(DISTINCT b.business_key), sum(b.amount_minor)
FROM attributed b
JOIN stg_terminal_service_tickets t
  ON t.terminal_system = b.terminal_system AND t.terminal_id = b.terminal_id
''')

In [ ]:
%sql
-- Step 1 is the baseline. Steps 2 and 3 multiply rows and money; distinct_payment_keys stays.
SELECT * FROM q1_source_exclusion ORDER BY step

In [ ]:
%sql
-- Second piece of evidence: several rows per merchant or terminal (n_rows against
-- distinct_parents), and first_delivery = last_delivery means the source arrives once and cannot
-- change the answer between C1 and C2.
SELECT 'merchant_campaigns' AS source_table, count(*) AS n_rows, count(DISTINCT business_key) AS distinct_keys,
       count(DISTINCT concat_ws('|', merchant_system, merchant_id)) AS distinct_parents,
       min(delivered_at) AS first_delivery, max(delivered_at) AS last_delivery
FROM stg_merchant_campaigns
UNION ALL
SELECT 'terminal_service_tickets', count(*), count(DISTINCT business_key),
       count(DISTINCT concat_ws('|', terminal_system, terminal_id)),
       min(delivered_at), max(delivered_at)
FROM stg_terminal_service_tickets

## Decisions for the close

The close uses seven of the nine sources. For each one, the list gives what a row means after cleaning, its key including the issuing system, and what the close takes from it.

**Included**

- **gateway_requests.** One row is one payment attempt. Key: gateway_system + gateway_reference. It gives the payment key, the payment day, the currency and the terminal.
- **processor_events.** One row is one version of one processor event. Key: processor_system + event_id, plus version. It gives the expected amount and the due time.
- **refund_events.** One row is one version of one refund. Key: processor_system + refund_id, plus version. It gives the completed refunds, which reduce the expected amount.
- **settlement_lines.** One row is one version of one provider posting. Key: provider_system + posting_id, plus version. It gives the settled amount and the provider-only postings.
- **reference_mappings.** One row maps one processor or provider reference to a payment. Key: mapping_system + mapping_id, plus version. It is looked up by from_system + from_reference and is the only link between the systems.
- **terminal_assignments.** One row is one assignment interval of a terminal. Key: terminal_system + assignment_id. It is looked up by terminal and request time and gives the acquiring bank.
- **delivery_manifests.** One row is one logical delivery. Key: delivery_id. It says when a row became available, which decides whether the row counts at a cutoff.

**Excluded**

- **merchant_campaigns.** One row is one campaign of a merchant and channel. Key: merchant_system + campaign_id. It has no payment key and is not a posting.
- **terminal_service_tickets.** One row is one maintenance ticket. Key: terminal_system + ticket_id. It has no payment key and no amount.

Both excluded sources describe merchants and terminals, not money. Joining them multiplies the payments, as section 3 shows.

**Assumptions that could change Finance's answer**

1. A posting is linked to a payment only through reference_mappings. Matching digits in two systems are not a link.
2. A fact delivered twice with identical business content counts as one fact, whichever delivery it arrived in.
3. A row that fails a check stays visible with its reason. It never drops out of a payment silently.

**Next:** notebook 03_clean_snapshots, which builds what is known at C1 and at C2.